# 🧠 ML Club Challenge: I Dropped an MNIST Neural Net

Welcome to the **MNIST Model Sorting Challenge**! This challenge is inspired by a machine learning puzzle originally designed by the quantitative trading firm **Jane Street**.

## 1. The Story & Challenge
You dropped a highly trained **MNIST Digit Classification Model** (a ResNet), and its **66 constituent linear layers** fell apart into an unlabelled bag of `.pth` weight files (`piece_0.pth` through `piece_65.pth`). 

Your objective is to **perfectly reassemble the network** to achieve a Mean Squared Error (MSE) of **exactly 0.0** on intermediate classification logits against the original model's predictions, restoring its high digit classification accuracy.

### 🧩 The Scrambled Pieces
All 66 layers are scrambled together in `data/pieces/`. However, they are easily isolated by their **weight tensor shapes**:
- **Front Projection Layer (`proj`):** Unique shape `(16, 784)`. Projects 784 flat image pixels of MNIST down to 16 feature dimensions.
- **Output Classification Layer (`last`):** Unique shape `(10, 16)`. Maps 16 features to 10 class logits (for digits 0 to 9).
- **Block Input Projections ($W_{\text{in}}$):** 32 pieces of shape `(32, 16)`.
- **Block Output Projections ($W_{\text{out}}$):** 32 pieces of shape `(16, 32)`.

### 📐 The Search Space
- **Pairing:** Matching each of the 32 input projections ($W_{\text{in}}$) to its corresponding output projection ($W_{\text{out}}$) ($32!$ combinations).
- **Ordering:** Sequencing the 32 reassembled blocks ($32!$ combinations).
- **Total joint combinations:** $(32!)^2 \approx 7.2 \times 10^{70}$, making brute-forcing completely impossible.

You must perform structural and mathematical analysis on the raw weight matrices to discover the pairing and sequencing signals left behind by gradient descent training!

--- 
## 2. Setup and Loading Data

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

# Load the historical dataset
df = pd.read_csv("data/historical_data.csv")
print(f"Historical dataset shape: {df.shape}")

# Extract MNIST pixel features (784 dimensions)
pixel_cols = [f"pixel_{i}" for i in range(28*28)]
X = torch.tensor(df[pixel_cols].values, dtype=torch.float32)

# Extract prediction logits (10 dimensions) from original model
logit_cols = [f"pred_logit_{i}" for i in range(10)]
target_logits = torch.tensor(df[logit_cols].values, dtype=torch.float32)

true_labels = torch.tensor(df["true_label"].values, dtype=torch.long)
print(f"Loaded {X.shape[0]} images. Features X shape: {X.shape} | Logits shape: {target_logits.shape}")

--- 
## 3. Sorting Scrambled Pieces by Shape
Let's scan `data/pieces/` and filter them by shape to isolate the components automatically.

In [ ]:
pieces_dir = "data/pieces"
piece_files = sorted([f for f in os.listdir(pieces_dir) if f.endswith(".pth")])

proj_layer = None
last_layer = None
inp_pieces = []
out_pieces = []

for filename in piece_files:
    filepath = os.path.join(pieces_dir, filename)
    piece = torch.load(filepath, map_location="cpu")
    weight = piece['weight']
    bias = piece['bias']
    
    if weight.shape == (16, 784):
        proj_layer = {'filename': filename, 'weight': weight, 'bias': bias}
    elif weight.shape == (10, 16):
        last_layer = {'filename': filename, 'weight': weight, 'bias': bias}
    elif weight.shape == (32, 16):
        inp_pieces.append({'filename': filename, 'weight': weight, 'bias': bias})
    elif weight.shape == (16, 32):
        out_pieces.append({'filename': filename, 'weight': weight, 'bias': bias})

print(f"Isolated Components:")
print(f"  - Front Projection Piece (`proj`): {proj_layer['filename']}")
print(f"  - Output Classification Piece (`last`): {last_layer['filename']}")
print(f"  - Input Block projections ($W_{{in}}$): {len(inp_pieces)} pieces")
print(f"  - Output Block projections ($W_{{out}}$): {len(out_pieces)} pieces")

--- 
## 4. Build and Evaluate Your Reassembled ResNet
Here is the helper function to calculate the Mean Squared Error (MSE) of your reassembled model's classification logits against the target logits in `historical_data.csv`.

In [ ]:
def evaluate_reconstruction(ordered_blocks):
    """
    Evaluates the Logits MSE of a candidate reassembled model.
    
    Arguments:
        ordered_blocks: A list of dictionaries representing the 32 blocks in proposed order.
                        Each block must contain 'inp_weight', 'inp_bias', 'out_weight', 'out_bias'.
    Returns:
        float: Logits MSE against target predictions.
    """
    curr_x = X.clone()
    # Front projection layer (784 -> 16)
    curr_x = curr_x @ proj_layer['weight'].t() + proj_layer['bias']
    
    # Forward pass through ordered blocks
    for b in ordered_blocks:
        w_in, b_in = b['inp_weight'], b['inp_bias']
        w_out, b_out = b['out_weight'], b['out_bias']
        curr_x = curr_x + torch.relu(curr_x @ w_in.t() + b_in) @ w_out.t() + b_out
    
    # Output layer (16 -> 10 logits)
    logits = curr_x @ last_layer['weight'].t() + last_layer['bias']
    
    mse = torch.mean((logits - target_logits) ** 2).item()
    return mse

--- 
## 5. Write Your Solver Here!
Implement your pairing and hill-climbing code below. Your goal is to achieve an MSE of exactly `0.0` and save your optimal configuration as `submission.csv` containing the header `block_index,inp_piece,out_piece`:

```csv
block_index,inp_piece,out_piece
0,piece_13.pth,piece_15.pth
1,piece_16.pth,piece_19.pth
...
31,piece_12.pth,piece_0.pth
```

Once saved, test your results by running `python verify_submission.py submission.csv` in your terminal!

In [ ]:
# TODO: Step 1. Analyze weight matrices to find the pairing signals and pair the blocks.
# TODO: Step 2. Analyze weight matrices to find sequencing proxies and establish a seed order.
# TODO: Step 3. Implement local search refinement to find the correct ordering.
# TODO: Step 4. Save your reassembled configuration as 'submission.csv' and run 'verify_submission.py'.